<a href="https://colab.research.google.com/github/MKMLLER/AllCups_AvitoPrice/blob/main/AllCups_3_MeanEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Импортируем библиотеки

In [1]:
!pip install --upgrade huggingface_hub transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 62.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1


In [2]:
!pip install torch torchvision catboost transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


In [3]:
import os
import random
import cv2
import re
import requests
from tqdm import tqdm

from PIL import Image
from IPython.display import display, Image
from PIL import Image as PILImage

import pandas as pd
import numpy as np
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn as nn
from torch import Tensor
from torch.optim import Adam

from transformers import AutoModel, AutoProcessor, AutoImageProcessor
from catboost import CatBoostRegressor, Pool

from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms
from torchvision.models.feature_extraction import create_feature_extractor

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_percentage_error as mape



In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp /content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/train_images.zip /content/train_images.zip
!unzip -q /content/train_images.zip -d /content/train_images_unzip

In [5]:
!cp /content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/test_images.zip /content/test_images.zip
!unzip -q /content/test_images.zip -d /content/test_images_unzip

# Получение эмбеддингов
- Рекомендую скачать папки с изображениями на свой google drive, чтобы не скачивать их с облака при каждом перезапуске сессии  

In [6]:
train_data = pd.read_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/train_dataset.parquet') # Табличные данные
test_data = pd.read_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/test_dataset.parquet')

In [7]:
class DataSet(Dataset): # Класс датасета, по индексу возращает преобразованную соответстующими трансформациями картинку
  def __init__(self, df, path, transform=None):
    self.df = df
    self.path = path
    self.transform = transform

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
        image_name = self.df.iloc[idx]['ID']

        # Создаем тензор сразу нужного размера (4 картинки, 3 канала, 224x224)
        final_images = torch.zeros(4, 3, 224, 224, dtype=torch.float32)
        mask = torch.zeros(4, dtype=torch.float32)

        for i in range(4):
            full_path = os.path.join(self.path, f'{image_name}_{i}.jpg')

            if os.path.exists(full_path):
                image = PILImage.open(full_path)

                if self.transform:
                    # Трансформируем картинку и СРАЗУ кладем в нужный индекс
                    final_images[i] = self.transform(image)

                mask[i] = 1.0

        return {
            'images': final_images,
            'mask': mask
        }

In [20]:
class cvModel(nn.Module):
  def __init__(self, model_path='content/dinov2-base'):
    super().__init__()

    self.backbone = AutoModel.from_pretrained(model_path)

  def forward(self, x):

    batch_size = x['images'].size(0)

    x_images = x['images'].view(-1, 3, 224, 224)
    x_mask = x['mask'].view(-1)

    valid_indices = torch.nonzero(x_mask).squeeze(1)
    outputs = torch.zeros(x_images.size(0), 768, device=x_images.device, dtype=x_images.dtype)

    valid_images = x_images[valid_indices]
    backbone_outputs = self.backbone(pixel_values=valid_images).pooler_output

    outputs[valid_indices] = backbone_outputs
    outputs = outputs.view(batch_size, 4, 768)

    mask = x['mask'].unsqueeze(-1)


    outputs_sum = torch.sum(outputs * mask, dim=1)

    mask_sum = torch.sum(mask, dim=1).clamp(min=1.0)

    return outputs_sum / mask_sum

In [9]:
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [19]:
# Устанавливаем git-lfs, если его нет
!git lfs install

# Клонируем только нужную модель (без истории коммитов, чтобы быстрее)
!git clone --depth 1 https://huggingface.co/facebook/dinov2-base content/dinov2-base

Git LFS initialized.
Cloning into 'content/dinov2-base'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), done.
Filtering content: 100% (2/2), 660.65 MiB | 28.93 MiB/s, done.


In [17]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [21]:
#processor = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
model = cvModel()
device = 'cuda'
model = model.to(device)

#train/test_images - папка на вашем диске
#train_dataset = DataSet(train_data, '/content/train_images_unzip', img_transform )
test_dataset = DataSet(test_data, '/content/test_images_unzip', img_transform )

'''train_loader = DataLoader(train_dataset,
    batch_size=64,          # Уменьшаем до 32, чтобы память не забивалась
    shuffle=False,
    num_workers=3,          # Убираем параллельные процессы CPU (убирает оверхед копирования)
    pin_memory=True)'''
test_loader = DataLoader(test_dataset,
                           batch_size=64,          # Уменьшаем до 32, чтобы память не забивалась
                            shuffle=False,
                            num_workers=0,          # Убираем параллельные процессы CPU (убирает оверхед копирования)
                            pin_memory=True)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

In [22]:
def get_embeddings(loader, model=model): # Достаем cls токены в качестве эмбеддингов картинок
  cls_tokens = []
  model.eval()
  with torch.no_grad():

    for batch in tqdm(loader):
      batch = {k : v.to(device) for k, v in batch.items()}


      outputs = model(batch)
      cls_tokens.append(outputs.cpu().numpy())

  return np.concatenate(cls_tokens, axis=0)



In [23]:
#train_embeddings = get_embeddings(train_loader)
test_embeddings = get_embeddings(test_loader)

100%|██████████| 391/391 [35:01<00:00,  5.38s/it]


In [24]:
#train_emb_df = pd.DataFrame(train_embeddings)
test_emb_df = pd.DataFrame(test_embeddings)
display(test_emb_df.shape)
#display(test_emb_df.shape)

(25000, 768)

In [25]:
#train_emb_df.to_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/train_embeddings.parquet')
test_emb_df.to_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/test_embeddings.parquet')

# Обучение полносвязной сети на предсказание цены

In [27]:
train_emb_df = pd.read_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/train_embeddings.parquet')
test_emb_df = pd.read_parquet('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/test_embeddings.parquet')
train_emb = train_emb_df.to_numpy()
test_emb = test_emb_df.to_numpy()


In [28]:
# Функция для корректной работы MultiLabelBinarizer
def put_zero(df):

  if len(df) == 0:
    return '0'

  if len(df) == 1 and (df[0] == None or str(df[0]).strip().lower() == 'none'):
    return '0'

  return df

# Раскрываем фичи вида ['1', '3', '2'] в отдельные
def add_features(df, mlb):
  mul_columns = train_data.filter(regex='mul')

  for column in mul_columns:
    df[column] = df[column].apply(put_zero)

  features = []
  for column in mul_columns:
    features.append(mlb.fit_transform(df[column])[:, 1:])
    df = df.drop([column], axis=1)

  features_concat = pd.DataFrame(np.concat(features, axis=1))
  out = pd.concat([df, features_concat], axis=1)

  return out


In [29]:
class PricePredictor(nn.Module):
    def __init__(self):
        super(PricePredictor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)



In [30]:
mlb = MultiLabelBinarizer()

train = add_features(train_data, mlb)
test = add_features(test_data, mlb)

y = np.log1p(train.pop('price_TARGET')) # Берем логарифм от целевой переменной чтобы уменьшить разброс и облегчить обучение

cat_features = list(train.select_dtypes(include='object').columns) # Категориальные фичи
X = train.fillna(0)

In [35]:
kfold = KFold(shuffle=False)

test_preds_folds = []
val_outputs = []

for i, (train_idx, test_idx) in enumerate(kfold.split(train_emb)):
    print(f"Split {i}")
    X_train, X_test, y_train, y_test = train_emb[train_idx], train_emb[test_idx], y.iloc[train_idx].values, y.iloc[test_idx].values

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)), batch_size=128, shuffle=True)

    model = PricePredictor().to('cuda')
    optim = Adam(model.parameters())
    criterion = nn.L1Loss()

    n_epochs = 20
    model.train()

    for epoch in range(n_epochs):
        for batch_x, batch_y in tqdm(train_loader):
          optim.zero_grad()
          batch_x = batch_x.to('cuda')
          batch_y = batch_y.to('cuda')

          preds = model(batch_x).squeeze()
          loss = criterion(preds, batch_y)
          loss.backward()

          optim.step()
        print(loss.item())

    model.eval()
    with torch.no_grad():
        X_val = torch.tensor(X_test, dtype=torch.float32).to('cuda')
        val_preds = (model(X_val).squeeze()).cpu().numpy()

        if 'pred_price' not in X.columns:
          X['pred_price'] = 0.0


        col_idx = X.columns.get_loc('pred_price')


        X.iloc[test_idx, col_idx] = np.expm1(val_preds)

    # На каждом шаге добавим предсказания моделью на валидации
        test_input = torch.tensor(test_emb).to('cuda')
        test_output = np.expm1(model(test_input).cpu().numpy())
        val_outputs.append(test_output)






Split 0


100%|██████████| 438/438 [00:02<00:00, 192.84it/s]


0.9715248942375183


100%|██████████| 438/438 [00:03<00:00, 134.16it/s]


0.970094621181488


...

0.8506856560707092


100%|██████████| 438/438 [00:01<00:00, 307.50it/s]


0.6285053491592407


100%|██████████| 438/438 [00:01<00:00, 311.90it/s]


0.7354118227958679


In [33]:
X_test

array([[-0.5163976 ,  0.23415178,  2.1168277 , ..., -1.82374   ,
         0.75506246,  1.9674554 ],
       [ 0.7777785 , -1.1777494 ,  1.8192168 , ..., -0.28043088,
         1.4383897 , -0.6649941 ],
       [ 0.8483999 ,  0.5574409 ,  3.333789  , ...,  0.11287603,
         1.1545105 ,  2.008137  ],
       ...,
       [ 0.02360968,  0.5055616 ,  1.8686577 , ..., -2.0744593 ,
        -0.27389437, -0.18488961],
       [ 0.4081911 , -0.928156  ,  1.7189426 , ..., -0.64434993,
        -1.8779416 ,  1.6554124 ],
       [-0.9350561 , -0.00445293,  0.9738588 , ...,  2.0661204 ,
         0.62171936,  1.5365622 ]], dtype=float32)

In [36]:
val_outputs_concat = np.concat(val_outputs, axis=1)
test_outputs_mean = np.mean(val_outputs_concat, axis=1)
test_outputs_mean


array([  55116.68,   85975.95, 1256950.  , ...,  800181.4 ,  210025.6 ,
        267309.34], dtype=float32)

# Обучение catboost на предсказанных полносвязным слоем фичах

---



In [37]:
X_train_catboost, X_test_catboost, y_train_catboost, y_test_catboost = train_test_split(X, y, test_size=0.2, random_state=42)

display(X_train_catboost['pred_price'])

train_pool = Pool(X_train_catboost, y_train_catboost, cat_features=cat_features)
test_pool = Pool(X_test_catboost, y_test_catboost, cat_features=cat_features)

catboost = CatBoostRegressor(10000,
                              learning_rate=0.15,
                              depth=6,
                              task_type='GPU',
                              loss_function='MAPE',
                              eval_metric='MAPE')

catboost.fit(train_pool,
              eval_set=test_pool,
              early_stopping_rounds=200,
              verbose=10)

,pred_price
47339,1.728317e+06
67456,5.190679e+05
12308,1.127028e+05
32557,1.088933e+05
664,1.021074e+06
...,...
37194,1.259130e+06
6265,1.742486e+06
54886,1.229852e+06
860,1.048253e+05


0:	learn: 0.0630949	test: 0.0618449	best: 0.0618449 (0)	total: 122ms	remaining: 20m 20s
10:	learn: 0.0573171	test: 0.0561161	best: 0.0561161 (10)	total: 401ms	remaining: 6m 3s
20:	learn: 0.0520349	test: 0.0508895	best: 0.0508895 (20)	total: 629ms	remaining: 4m 58s
30:	learn: 0.0473230	test: 0.0462594	best: 0.0462594 (30)	total: 855ms	remaining: 4m 35s
40:	learn: 0.0431803	test: 0.0421737	best: 0.0421737 (40)	total: 1.07s	remaining: 4m 19s
50:	learn: 0.0395739	test: 0.0386134	best: 0.0386134 (50)	total: 1.28s	remaining: 4m 9s
60:	learn: 0.0365165	test: 0.0356078	best: 0.0356078 (60)	total: 1.49s	remaining: 4m 3s
70:	learn: 0.0339136	test: 0.0330726	best: 0.0330726 (70)	total: 1.7s	remaining: 3m 58s
80:	learn: 0.0317045	test: 0.0309337	best: 0.0309337 (80)	total: 1.93s	remaining: 3m 56s
90:	learn: 0.0298745	test: 0.0291794	best: 0.0291794 (90)	total: 2.14s	remaining: 3m 53s
100:	learn: 0.0283698	test: 0.0277411	best: 0.0277411 (100)	total: 2.35s	remaining: 3m 50s
110:	learn: 0.0271639	te

CatBoostRegressor(depth=6, eval_metric='MAPE', iterations=10000, learning_rate=0.15, loss_function='MAPE', task_type='GPU')

In [38]:

preds = np.expm1(catboost.predict(test_pool)) # Берем экспоненту от предсказаний, чтобы вернуть их в требуемый вид
print(f'Метрика MAPE: {mape(np.expm1(y_test), preds)}')

Метрика MAPE: 2.106935214163872


# Предсказание цен и отправка сабмита
 - Прогон тестовых эмбедингов через полносвязный слой, подача теста в катбуст

In [42]:
# По первой оси склеиваем предсказания полносвязной модели и фичи для catboost
X_test = pd.concat([test.fillna(0), pd.DataFrame({ 'pred_price' : test_outputs_mean})], axis=1)

test_preds = catboost.predict(X_test)
test_preds_exp = np.expm1(test_preds)

In [43]:
test_preds_exp.shape

(25000,)

In [44]:
submission1 = pd.DataFrame(
    {
        'ID' : test['ID'],
        'target' : test_preds_exp
    }
)
submission1.to_csv('/content/drive/MyDrive/ML_Notebooks_and_projects/AllCups_3/Data/submission11.csv', index=False)

# Перебор параметров катбуста

In [ ]:
'''param_grid = {
    'depth': [6, 8, 12],
    'l2_leaf_reg': [3, 5, 7],
    'learning_rate': [0.03, 0.1, 0.2]
}

catboost2 = CatBoostRegressor(1500,
                              learning_rate=0.1,
                              task_type='GPU',
                              loss_function='MAPE',
                              eval_metric='MAPE',
                              cat_features=cat_features)

grid_search_result = catboost2.grid_search(
    param_grid,
    X=X_train_catboost,
    y=y_train_catboost,
    cv=2,
    partition_random_seed=42,
)'''